# Cenário 5: COMPAS e sub-representação racial no treino

Seminário de Viés (P1). Se um grupo racial aparece pouco, ou de forma distorcida, nos dados de treino, o modelo passa a errar de outro jeito para esse grupo?

Usamos o COMPAS (ProPublica, Broward County) para prever `two_year_recid`, ou seja, se a pessoa reincidiu em até dois anos. Comparamos afro-americanos e caucasianos, porque os outros grupos são pequenos demais. Um dos grupos é reduzido a 10% **só no treino**, e o teste fica intacto.

| Pilar do enunciado | Onde |
|---|---|
| 1. Explicar o código e discutir ética | seções 1 e 5 |
| 2. Onde nasce o desequilíbrio | seção 2 |
| 3. Mecanismo do viés | seções 3 e 4 |
| 4. Mitigação | fora deste notebook |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

TARGET = "two_year_recid"
NUMERIC = ["age", "priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count"]
CATEGORICAL = ["sex", "c_charge_degree"]

GROUP_SUB = "African-American"
GROUP_REF = "Caucasian"
GROUPS = [GROUP_SUB, GROUP_REF]
FRAC = 0.10
N_SEEDS = 30

In [ ]:
def rates(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    pos, neg = tp + fn, fp + tn
    return {
        "n": len(y),
        "taxa_base": np.mean(y),
        "TPR": tp / pos if pos else np.nan,
        "FNR": fn / pos if pos else np.nan,
        "FPR": fp / neg if neg else np.nan,
        "precisao": tp / (tp + fp) if (tp + fp) else np.nan,
        "acuracia": (tp + tn) / len(y),
    }


def metrics_by_group(y, pred, race):
    out = {g: rates(y[race == g], pred[race == g]) for g in GROUPS}
    out["Geral"] = rates(y, pred)
    return pd.DataFrame(out).T.rename_axis("grupo")


def features(use_race):
    return NUMERIC + CATEGORICAL + (["race"] if use_race else [])


def build_model(model, use_race):
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(drop="if_binary"), features(use_race)[len(NUMERIC):]),
    ])
    if model == "forest":
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
    else:
        clf = LogisticRegression(max_iter=1000)
    return Pipeline([("pre", pre), ("clf", clf)])


def split(df, seed):
    strat = df.race + "_" + df[TARGET].astype(str)
    return train_test_split(df, test_size=0.30, stratify=strat, random_state=seed)

## 1. Contexto e dados

O COMPAS estima o risco de reincidência e apoia decisões como a liberdade provisória. Em 2016 o ProPublica ("Machine Bias") mostrou que os erros não eram iguais entre os grupos. Entre quem não reincidiu, afro-americanos eram classificados como alto risco com mais frequência (FPR maior). Entre quem reincidiu, brancos eram classificados como baixo risco com mais frequência (FNR maior). A Northpointe respondeu que o escore é calibrado entre os grupos.

Os dois argumentos podem valer ao mesmo tempo, porque com taxas-base diferentes não dá para ter calibração e igualdade de erros juntas (Chouldechova, 2017; Kleinberg et al., 2016).

Aqui não reproduzimos esse viés. Simulamos outra origem, a coleta de dados: o modelo é o mesmo, muda só o que entra no treino.

In [ ]:
URL = ("https://raw.githubusercontent.com/propublica/compas-analysis/"
       "master/compas-scores-two-years.csv")
raw = pd.read_csv(URL)

df = raw[
    raw.days_b_screening_arrest.between(-30, 30)
    & (raw.is_recid != -1)
    & (raw.c_charge_degree != "O")
    & (raw.score_text != "N/A")
].copy()
df = df[df.race.isin(GROUPS)].reset_index(drop=True)

print(f"{len(raw)} linhas brutas -> {len(df)} depois dos filtros")
df.groupby("race")[TARGET].agg(n="size", taxa_reincidencia="mean")

Para referência, o erro do próprio COMPAS nesses dados (Medium e High contam como alto risco, como no ProPublica):

In [ ]:
compas_high = (df.score_text != "Low").astype(int).to_numpy()
metrics_by_group(df[TARGET].to_numpy(), compas_high, df.race.to_numpy())

## 2. Onde nasce o desequilíbrio

O treino (70%) e o teste (30%) são separados antes de qualquer filtro, estratificando por grupo e por alvo. Depois, só no treino, o grupo escolhido é reduzido a `FRAC` (10%) por uma das formas de `SAMPLERS`. O resto do treino fica inteiro. A função `make_biased_train` é a etapa em que o desequilíbrio é criado.

As formas de amostrar são:

- **aleatorio:** sorteio simples.
- **poucos_antecedentes / muitos_antecedentes:** quem tem menos (ou mais) `priors_count` tem mais chance de entrar.
- **reincidentes_x3 / x10:** quem reincidiu tem 3 (ou 10) vezes mais chance de entrar. É uma seleção pelo desfecho e uma suposição nossa, parecida com uma base que só registra quem voltou a ser preso.

In [ ]:
def weighted_sampler(weight_fn):
    def sample(group, seed):
        w = np.asarray(weight_fn(group), dtype=float)
        idx = np.random.default_rng(seed).choice(
            len(group), size=round(FRAC * len(group)), replace=False, p=w / w.sum()
        )
        return group.iloc[idx]
    return sample


SAMPLERS = {
    "aleatorio": lambda group, seed: group.sample(frac=FRAC, random_state=seed),
    "poucos_antecedentes": weighted_sampler(lambda g: 1 / (1 + g.priors_count) ** 2),
    "muitos_antecedentes": weighted_sampler(lambda g: (1 + g.priors_count) ** 2),
    "reincidentes_x3": weighted_sampler(lambda g: np.where(g[TARGET] == 1, 3, 1)),
    "reincidentes_x10": weighted_sampler(lambda g: np.where(g[TARGET] == 1, 10, 1)),
}


def make_biased_train(train, seed, sub=GROUP_SUB, sampling="aleatorio"):
    # >>> aqui nasce o desequilíbrio <<<
    is_sub = train.race == sub
    kept = SAMPLERS[sampling](train[is_sub], seed)
    return pd.concat([kept, train[~is_sub]])

In [ ]:
SEED = 42
train, test = split(df, SEED)
biased = make_biased_train(train, SEED)

comp = pd.DataFrame({"n_completo": train.race.value_counts(),
                     "n_enviesado": biased.race.value_counts()})
comp["%_completo"] = 100 * comp.n_completo / comp.n_completo.sum()
comp["%_enviesado"] = 100 * comp.n_enviesado / comp.n_enviesado.sum()
print(comp.round(1))
print(f"\nTaxa de reincidência no treino: completo = {train[TARGET].mean():.3f}, "
      f"enviesado = {biased[TARGET].mean():.3f}")

ax = comp[["%_completo", "%_enviesado"]].T.plot.bar(rot=0, figsize=(6, 3.5))
ax.set_ylabel("% do treino")
ax.set_title("Composição do treino antes e depois do filtro")
plt.tight_layout()
plt.show()

O que cada forma de amostrar faz com os afro-americanos que sobram (seed 42):

In [ ]:
def kept_profile(sampling):
    b = make_biased_train(train, SEED, sampling=sampling)
    return b[b.race == GROUP_SUB][["priors_count", TARGET]].mean()


profile = {name: kept_profile(name) for name in SAMPLERS}
profile["grupo completo"] = train[train.race == GROUP_SUB][["priors_count", TARGET]].mean()
pd.DataFrame(profile).T.rename(columns={TARGET: "taxa_reincidencia"})

## 3. Mecanismo do viés: um exemplo

Treinamos dois modelos iguais (regressão logística, sem `race`), um com o treino completo e outro com o treino filtrado por sorteio. Os dois são avaliados no mesmo teste.

Vemos duas explicações possíveis:

- **H1:** o modelo aprende pior o grupo reduzido, e só ele piora.
- **H2:** reduzir o grupo muda a taxa de reincidência do treino, o modelo reajusta o risco "médio" e o corte de 0,5, e os dois grupos se deslocam juntos.

In [ ]:
cols = features(False)
base = build_model("logistic", False).fit(train[cols], train[TARGET])
bias = build_model("logistic", False).fit(biased[cols], biased[TARGET])

y_test, race_test = test[TARGET].to_numpy(), test.race.to_numpy()
pd.concat({
    "baseline": metrics_by_group(y_test, base.predict(test[cols]), race_test),
    "enviesado": metrics_by_group(y_test, bias.predict(test[cols]), race_test),
}, names=["modelo"])[["n", "taxa_base", "TPR", "FNR", "FPR", "precisao", "acuracia"]]

Probabilidade média prevista por grupo. A queda aparece só no grupo reduzido (H1) ou nos dois (H2)? Mudanças pequenas aqui já mexem bastante em TPR e FPR, porque muitos casos ficam perto do corte de 0,5. Esta seção usa uma só semente, e a próxima repete tudo várias vezes.

In [ ]:
p_base = pd.Series(base.predict_proba(test[cols])[:, 1], index=test.index)
p_bias = pd.Series(bias.predict_proba(test[cols])[:, 1], index=test.index)
pd.DataFrame({
    "taxa real": test.groupby("race")[TARGET].mean(),
    "prob. média, baseline": p_base.groupby(test.race).mean(),
    "prob. média, enviesado": p_bias.groupby(test.race).mean(),
})

## 4. Todos os casos

Cada caso treina um **baseline** (treino completo) e um modelo **enviesado** (treino filtrado), avalia os dois no mesmo teste e se repete com `N_SEEDS` sementes. Os casos variam o modelo, o uso de `race` como feature, o grupo reduzido e a forma de amostrar. O grupo reduzido aparece como `reduzido` e o outro como `outro`.

In [ ]:
CASES = {
    "A1 sorteio | logística": dict(),
    "A2 sorteio | logística, reduz caucasianos": dict(sub=GROUP_REF),
    "A3 sorteio | floresta": dict(model="forest"),
    "A4 sorteio | logística + race": dict(use_race=True),
    "A5 sorteio | floresta + race": dict(model="forest", use_race=True),
    "B1 poucos antecedentes | logística": dict(sampling="poucos_antecedentes"),
    "B2 poucos antecedentes | floresta + race": dict(sampling="poucos_antecedentes", model="forest", use_race=True),
    "C1 muitos antecedentes | logística": dict(sampling="muitos_antecedentes"),
    "C2 muitos antecedentes | floresta + race": dict(sampling="muitos_antecedentes", model="forest", use_race=True),
    "D1 reincidentes x3 | logística": dict(sampling="reincidentes_x3"),
    "D2 reincidentes x10 | logística": dict(sampling="reincidentes_x10"),
    "D3 reincidentes x3 | floresta + race": dict(sampling="reincidentes_x3", model="forest", use_race=True),
    "D4 reincidentes x10 | floresta + race": dict(sampling="reincidentes_x10", model="forest", use_race=True),
}


def run_case(case, seed, model="logistic", use_race=False, sampling="aleatorio", sub=GROUP_SUB):
    cols = features(use_race)
    train, test = split(df, seed)
    biased = make_biased_train(train, seed, sub, sampling)
    y, race = test[TARGET].to_numpy(), test.race.to_numpy()

    rows = []
    for label, data in [("baseline", train), ("enviesado", biased)]:
        fitted = build_model(model, use_race).fit(data[cols], data[TARGET])
        out = metrics_by_group(y, fitted.predict(test[cols]), race).reset_index()
        out["papel"] = np.where(out.grupo == sub, "reduzido", np.where(out.grupo == "Geral", "geral", "outro"))
        out["caso"], out["modelo"], out["seed"] = case, label, seed
        rows.append(out)
    return pd.concat(rows, ignore_index=True)


res = pd.concat(
    [run_case(case, seed, **spec) for case, spec in CASES.items() for seed in range(N_SEEDS)],
    ignore_index=True,
)

Taxas médias por grupo, antes e depois do filtro:

In [ ]:
by_group = res[res.papel != "geral"].pivot_table(
    index=["caso", "papel"], columns="modelo", values=["FNR", "FPR"]
)
by_group = by_group.reindex(pd.MultiIndex.from_product([list(CASES), ["reduzido", "outro"]]))
by_group[[("FNR", "baseline"), ("FNR", "enviesado"), ("FPR", "baseline"), ("FPR", "enviesado")]].round(3)

Variação (enviesado menos baseline), média das sementes e desvio-padrão entre parênteses. A coluna `diferença` é a variação do grupo reduzido menos a do outro grupo. Se a variação é pequena perto do desvio, pode ser só ruído.

In [ ]:
wide = res[res.papel != "geral"].pivot_table(
    index=["caso", "seed", "papel"], columns="modelo", values=["FNR", "FPR"]
)
delta = (wide.xs("enviesado", axis=1, level=1) - wide.xs("baseline", axis=1, level=1)).unstack("papel")
for m in ["FNR", "FPR"]:
    delta[(m, "diferença")] = delta[(m, "reduzido")] - delta[(m, "outro")]

order = [(m, p) for m in ["FNR", "FPR"] for p in ["reduzido", "outro", "diferença"]]
mean = delta.groupby("caso").mean()[order].reindex(list(CASES))
std = delta.groupby("caso").std()[order].reindex(list(CASES))
mean.map("{:+.3f}".format) + " (" + std.map("{:.3f}".format) + ")"

Os mesmos resultados em gráficos, um bloco por família e uma linha por caso. À esquerda está o **FNR** (reincidentes que passaram batido) e à direita o **FPR** (não reincidentes marcados como alto risco). Em cada gráfico, a primeira barra de cada cor é o modelo com o treino completo (baseline) e a segunda é o modelo com o treino filtrado (enviesado). A legenda diz qual grupo foi reduzido.

In [ ]:
FAMILIES = {"A": "A. Sorteio aleatório", "B": "B. Poucos antecedentes",
            "C": "C. Muitos antecedentes", "D": "D. Reincidentes favorecidos"}
COLORS = {GROUP_SUB: "tab:blue", GROUP_REF: "tab:orange"}
MODELS = ["baseline", "enviesado"]

stats = res[res.papel != "geral"].groupby(["caso", "grupo", "modelo"])[["FNR", "FPR"]].agg(["mean", "std"])


def plot_family(letter):
    cases = [c for c in CASES if c[0] == letter]
    fig, axes = plt.subplots(len(cases), 2, figsize=(11, 2.9 * len(cases)), squeeze=False)
    for row, case in zip(axes, cases):
        reduced = CASES[case].get("sub", GROUP_SUB)
        for ax, metric in zip(row, ["FNR", "FPR"]):
            for k, group in enumerate(GROUPS):
                mean = [stats.loc[(case, group, m), (metric, "mean")] for m in MODELS]
                std = [stats.loc[(case, group, m), (metric, "std")] for m in MODELS]
                x = np.arange(2) + (k - 0.5) * 0.38
                label = f"{group} (reduzido)" if group == reduced else group
                ax.bar(x, mean, 0.38, yerr=std, capsize=3, color=COLORS[group], label=label)
                for xi, v, e in zip(x, mean, std):
                    ax.text(xi, v + e + 0.02, f"{v:.2f}", ha="center", fontsize=7)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(["baseline\n(treino completo)", "enviesado\n(treino filtrado)"], fontsize=8)
            ax.set_ylim(0, 1.15)
            ax.set_ylabel(metric)
        row[0].set_title(case, loc="left", fontweight="bold", fontsize=10)
        row[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    fig.suptitle(FAMILIES[letter], fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 0.82, 0.97))
    plt.show()


for letter in FAMILIES:
    plot_family(letter)

## 5. Discussão

### O que encontramos

Os valores são aproximados e vêm de uma execução de referência com 30 sementes. As tabelas da seção 4 valem mais do que estes números. Variação entre parênteses é o desvio-padrão entre sementes.

**Sorteio aleatório (A)**

- O baseline já é desigual, mesmo sem filtro: FPR de ~0,31 nos afro-americanos contra ~0,15 nos caucasianos, e FNR de ~0,32 contra ~0,59. Isso vem das taxas-base diferentes (~0,52 e ~0,39), não da amostragem.
- Reduzir os afro-americanos (A1) piora a detecção neles: o FNR sobe ~0,09 (0,03) e o FPR cai ~0,07. Mas o outro grupo também se desloca no mesmo sentido (FNR +0,06, FPR -0,04), então a diferença entre os grupos quase não muda (FNR +0,02, dentro do ruído). É o que esperaríamos de H2: reduzir o grupo muda a taxa de reincidência do treino e o modelo reajusta o risco para todo mundo.
- Reduzir os caucasianos (A2) empurra tudo no sentido oposto: o modelo passa a prever risco mais alto para os dois grupos (FPR +0,03 e +0,04).
- Floresta e `race` como feature (A3 a A5) seguem o mesmo padrão. Com `race` a variação entre sementes aumenta bastante (A4: FNR +0,08 com desvio de 0,09), porque o modelo estima um coeficiente de raça com só ~220 exemplos do grupo.

**Coleta enviesada por antecedentes (B e C)**

- Sem `race` (B1, C1) o resultado é igual ao do sorteio: os dois grupos se deslocam juntos.
- Com floresta e `race` o efeito passa a ser específico do grupo reduzido, e o sentido depende de quem sobra na amostra. Se sobram afro-americanos com poucos antecedentes (B2), o FNR deles sobe ~0,19 e o FPR cai ~0,13, enquanto o outro grupo quase não muda. A diferença de FNR entre os grupos sobe ~0,15 (0,06). Se sobram os com muitos antecedentes (C2), inverte: o FNR cai ~0,16 e o FPR sobe ~0,27.

**Preferência por reincidentes (D)**

Nesta forma de amostrar, os afro-americanos que sobram reincidem muito mais que o grupo real: ~0,77 (x3) e ~0,94 (x10), contra ~0,52.

- Na logística sem `race` (D1, D2) o FPR do grupo reduzido sobe (+0,03 e +0,09), mas o do outro grupo também (+0,02 e +0,06). A diferença entre grupos quase não muda.
- Na floresta com `race` (D3, D4) o modelo aprende que pertencer ao grupo reduzido significa alto risco: o FPR sobe ~0,56 (x3) e ~0,68 (x10), e o do outro grupo fica parado. Quase todo afro-americano passa a ser marcado como alto risco, mesmo sem ter reincidido.

### Leitura

Os casos se dividem em dois regimes.

1. **Deslocamento global.** Quando o modelo não tem como aprender uma regra por grupo (sorteio, ou logística sem `race`), reduzir um grupo só muda a taxa-base do treino. Os dois grupos se deslocam juntos e a disparidade quase não muda, embora a detecção do grupo reduzido piore.
2. **Regra por grupo.** Quando o modelo enxerga `race` e a amostra do grupo difere do grupo real, ele aprende essa diferença como regra, e a disparidade cresce muito.

`race` sozinha não cria o viés: com dados completos ou sorteio (A4, A5) a mudança é pequena. O viés vem da amostra distorcida, e `race` é o canal que o transforma em tratamento desigual. Sem `race` ainda há efeito, mais fraco (B1, C1, D1, D2), provavelmente por proxies como antecedentes e idade.

### Limites

- A distorção é simulada. O viés do COMPAS real tem outras origens (dados históricos, features, modelo proprietário).
- Os desenhos de amostragem B, C e D são escolhas nossas, e D é uma seleção pelo desfecho. Os resultados mostram o que cada desenho produz e não que a coleta real seja assim.
- O rótulo `two_year_recid` mede nova prisão, não nova ofensa. Ele herda o viés de policiamento, então o "gabarito" já é desigual.
- Só dois grupos, um corte de 10%, um conjunto de dados e dois modelos.

### Questões éticas

- **O modelo não tem intenção.** Ele reproduz o padrão da amostra. O problema é o resultado: pessoas tratadas de forma diferente por pertencerem a um grupo.
- **Qual erro é o dano?** Um FPR maior mantém alguém preso sem necessidade, e um FNR maior solta alguém que reincide. Em B2 o grupo reduzido tem FNR maior e FPR menor, em C2 e D3 o contrário. Dizer que o grupo foi "prejudicado" depende de qual erro se considera.
- **Proxies.** Mesmo sem `race`, `priors_count` e `age` carregam informação racial por causa de históricos desiguais de abordagem e prisão. Tirar a coluna não deixa o modelo cego à raça.
- **Usar o grupo na decisão.** Colocar `race` como feature é tratamento desigual, e é o que torna o efeito mais forte nos casos B2, C2, D3 e D4.